# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset summarizes ordered logistic regression results for adoption predictors of indigenous and modern knowledge in rangeland management across pastoralist households in Kenya.

### Dataset Source
The dataset is described via a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
pd.set_option('display.max_columns', None)

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
meta = dataset.metadata
print(f"Dataset: {meta.name}")
print(f"Description: {meta.description}")

## 2. Data Overview
Explore available record sets, fields, and their `@id` values.

**Note:** All entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# List all available record sets and their IDs
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets detected in schema (recordSet is empty in metadata). Attempting to enumerate via dataset.record_sets...")
    # Get the underlying record_set ids from dataset.record_sets
    record_sets = list(dataset.record_sets)
else:
    print("Record sets declared in schema:")
    for rec in record_sets:
        print(f"- {rec}")

# Explore fields and columns for each record set
for rec in record_sets:
    print(f"\nRecord set @id: {rec}")
    fields = dataset.fields(record_set=rec)
    for fld in fields:
        print(f"  Field @id: {fld['@id']} | name: {fld.get('name', '-')}")
        if 'column' in fld:
            col = fld['column']
            if isinstance(col, dict):
                print(f"    Column @id: {col.get('@id', '-')}, name: {col.get('name', '-')}")
            elif isinstance(col, list):
                for c in col:
                    print(f"    Column @id: {c.get('@id', '-')}, name: {c.get('name', '-')}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame using `mlcroissant`. Use only the record set `@id`s for references.

In [ ]:
# Extract all data from available record sets
dataframes = {}
print(f"Found {len(record_sets)} record set(s): {record_sets}")

for record_set_id in record_sets:
    try:
        records_iter = dataset.records(record_set=record_set_id)
        df = pd.DataFrame(records_iter)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")
        print(f"Columns: {list(df.columns)}\n")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Inspect the first few rows of each DataFrame
for record_set_id, df in dataframes.items():
    print(f"\nData preview for record set {record_set_id}:")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Process and analyze numeric fields (referenced by `@id`) in a chosen record set.

*Example steps: Filtering by threshold, normalization, grouping by a key attribute.*

In [ ]:
# Choose a record set and field for demonstration
if dataframes:
    # For demonstration, choose the first available record set and try to find numerical fields
    example_record_set = list(dataframes.keys())[0]
    example_df = dataframes[example_record_set]

    # Print columns and their types
    print(f"Columns in {example_record_set}: {example_df.columns.tolist()}")

    # Identify numeric columns by pandas dtype
    numeric_fields = example_df.select_dtypes(include='number').columns.tolist()
    if not numeric_fields:
        print("No numeric fields found for EDA.")
    else:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")

        # Filtering
        threshold = example_df[numeric_field].mean() if not pd.isna(example_df[numeric_field].mean()) else 0
        threshold = float(threshold)  # In case it's numpy.float64
        filtered_df = example_df[example_df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Grouping by the first non-numeric field
        group_fields = [col for col in example_df.columns if col not in numeric_fields]
        if group_fields:
            group_field = group_fields[0]
            print(f"Grouping by field: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No non-numeric field available to group by.")
else:
    print("No loaded DataFrames to analyze.")

## 5. Visualization
Visualize distribution and relationships between numeric fields and group/key fields.

In [ ]:
# Visualization example, if numeric field exists
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and 'numeric_field' in locals():
    plt.figure(figsize=(8, 5))
    sns.histplot(example_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}' in record set {example_record_set}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # If grouping field exists, visualize mean by group
    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field, ci=None)
        plt.title(f"Mean {numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated:
- Loading a Croissant dataset and its metadata with `mlcroissant`.
- Enumerating record sets and fields by their `@id`.
- Extracting data to pandas DataFrames for practical analysis.
- Performing EDA including filtering, normalization, and group-by with explicit `@id` references.
- Visualizing numerical variables and grouped means.

**For further research:**
- Dive into specific field `@id`s from the schema documentation for richer contextual analysis.
- Consider more advanced statistical methods or visualizations suited to your use-case and the dataset's structure.
- Reference the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) for additional utilities and complex schema navigation.
